# TissueSpectF · Colab

Demostración técnica del pipeline de espectros transcriptómicos de hígado:
instala, descarga cohortes GEO, audita las etiquetas, calcula espectros y
espectros característicos por condición, y evalúa baselines.

> **Qué es y qué no.** Se omite `maxt` (horas en 2 cores) y, en modo rápido, se
> restringe el genoma. Sirve para verificar el flujo y las etiquetas, **no para
> producir resultados**. Los parámetros de producción están en `RUNBOOK.md`.

## 1 · Configuración reproducible

In [ ]:
import os
import sys
import shutil
import subprocess
import time
from pathlib import Path

REPO_URL = "https://github.com/Danpc11/TissueSpectF.git"
REPO_REF = "main"          # Para una tesis: fija un tag o un commit.
REPO_DIR = Path("/content/TissueSpectF")

DATASETS = ["GSE130970", "GSE135251", "GSE162694", "GSE142530"]

# FAST_MODE gobierna de verdad el costo. Con dos cores, el consenso sobre 23
# cromosomas y 7 clases tarda horas: el nulo de permutación recalcula un
# espectro consenso completo por sorteo. Restringir el genoma y bajar los
# sorteos es lo que hace que este notebook termine.
FAST_MODE = True
CHROMOSOMES = "1,17" if FAST_MODE else None   # None = grid completo
N_NULL      = 99 if FAST_MODE else 199        # 999+ para publicación

DATA_DIR, INTERIM_DIR, RESULTS_DIR = map(
    Path, ("/content/data", "/content/interim", "/content/results"))
for d in (DATA_DIR, INTERIM_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

os.environ.update({"TSF_GEO_DIR": str(DATA_DIR),
                   "TSF_INTERIM_DIR": str(INTERIM_DIR),
                   "TSF_RESULTS_DIR": str(RESULTS_DIR)})

def tsf(*args, scope=True):
    """Llama al CLI. `scope` aplica la restricción de cromosomas del modo rápido."""
    cmd = ["./tsf", *args]
    if scope and CHROMOSOMES:
        cmd += ["--chromosomes", CHROMOSOMES]
    t0 = time.time()
    subprocess.run(cmd, check=True)
    print(f"[{' '.join(args[:2])}] {time.time() - t0:.0f} s")

print("Datasets   :", ", ".join(DATASETS))
print("Cromosomas :", CHROMOSOMES or "grid completo")
print("n_null     :", N_NULL)
print("Resultados :", RESULTS_DIR)

In [ ]:
if shutil.which("Rscript") is None:
    print("Installing R...")
    subprocess.run(
        ["apt-get", "-qq", "update"],
        check=True,
    )
    subprocess.run(
        ["apt-get", "-qq", "install", "-y", "r-base-core"],
        check=True,
    )

print(subprocess.run(
    ["Rscript", "--version"],
    capture_output=True,
    text=True,
    check=True,
).stderr.strip())

In [ ]:
if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    print("Repository already present; reusing", REPO_DIR)

os.chdir(REPO_DIR)
subprocess.run(["chmod", "+x", "tsf"], check=True)
commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
print("Repository commit:", commit)
subprocess.run(["./tsf", "--help"], check=True)

### Guardar resultados en Google Drive (opcional)

Ejecuta la celda siguiente **antes del análisis** si quieres conservar los resultados al cerrar Colab. Déjala sin ejecutar para usar el almacenamiento temporal.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# RESULTS_DIR = Path("/content/drive/MyDrive/TissueSpectF/results")
# RESULTS_DIR.mkdir(parents=True, exist_ok=True)
# os.environ["TSF_RESULTS_DIR"] = str(RESULTS_DIR)
# print("Persistent results:", RESULTS_DIR)

## 2 · Verificación del código

In [ ]:
# Núcleo estadístico y reglas de etiquetado.
subprocess.run(["make", "test"], check=True)

In [ ]:
# Dependencias mínimas para las pruebas y baselines Python; no instala PyTorch.
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "pytest", "numpy", "pandas", "scipy", "scikit-learn", "pyyaml",
], check=True)
subprocess.run([sys.executable, "-m", "pytest", "tests/ml", "-q"], check=True)

In [ ]:
# End-to-end sintético con pocas permutaciones.
env = os.environ.copy()
env.update({"TSF_MAXT_B": "100", "TSF_CONDITION_B": "300"})
subprocess.run(["./tsf", "selfcheck"], env=env, check=True)

## 3 · Descarga y auditoría de entradas

Se descargan únicamente las cohortes declaradas en `DATASETS`. El notebook no usa `./tsf check` porque actualmente ese comando revisa todas las configuraciones del repositorio, incluso las cohortes no seleccionadas.

In [ ]:
tsf("fetch", *DATASETS, "--geo-dir", str(DATA_DIR), scope=False)

# `check` confirma que cada archivo declarado existe con el nombre que espera el
# config, y nombra los parecidos cuando no. GEO renombra los suplementarios.
tsf("check", "--geo-dir", str(DATA_DIR), scope=False)

for path in sorted(DATA_DIR.glob("*")):
    print(f"{path.name:65s} {path.stat().st_size / 1024**2:8.1f} MB")

In [ ]:
# El inspector cruza DOS campos: es esa tabla la que distingue dos grupos que
# un paper llama igual. Cambia el accession y los dos campos para ver otro.
subprocess.run([
    "Rscript", "scripts/inspect_series_matrix.R",
    str(DATA_DIR / "GSE130970_series_matrix.txt.gz"),
    "fibrosis", "steatosis",
], check=True)

## 4 · Ingesta y auditoría de etiquetas

Esta es la revisión más importante antes de calcular espectros. Una etiqueta equivocada produce un espectro característico de la población equivocada.

In [ ]:
tsf("ingest", *DATASETS)

In [ ]:
import pandas as pd
from IPython.display import display

audits = []
for dataset in DATASETS:
    path = INTERIM_DIR / dataset / "label_audit.tsv"
    frame = pd.read_csv(path, sep="\t")
    frame["dataset_id"] = dataset
    audits.append(frame)

audit = pd.concat(audits, ignore_index=True)
audit["keep"] = audit["keep"].astype(str).str.lower().isin(["true", "t", "1"])

kept = audit[audit["keep"]]
label_counts = (
    kept.groupby(["dataset_id", "class_id"], dropna=False)
        .size()
        .unstack(fill_value=0)
)
display(label_counts)

unresolved = audit[~audit["keep"]][
    ["dataset_id", "sample_id", "condition", "label_rule"]
]
print("Excluded or unresolved samples:", len(unresolved))
display(unresolved.head(20))

# Sanity check already established for GSE130970.
n_normal_130970 = len(kept[
    (kept["dataset_id"] == "GSE130970") &
    (kept["condition"] == "Normal_histology")
])
assert n_normal_130970 == 6, f"Expected 6 Normal_histology; observed {n_normal_130970}"
print("GSE130970 Normal_histology: 6/6 ✓")

In [ ]:
# Cobertura del grid: valores muy distintos entre cohortes requieren revisión.
coverage = []
for dataset in DATASETS:
    path = INTERIM_DIR / dataset / "grid_coverage.tsv"
    frame = pd.read_csv(path, sep="\t")
    frame["dataset_id"] = dataset
    coverage.append(frame)

coverage = pd.concat(coverage, ignore_index=True)
display(coverage.pivot(index="chr", columns="dataset_id", values="coverage_pct"))

## 5 · Espectros individuales y por condición

El CLI actual procesa el grid cromosómico completo. Los genes ausentes mantienen su posición y se tratan como no observados; no se rellenan con cero ni con `NA` dentro de una FFT convencional.

In [ ]:
tsf("spectra", *DATASETS)

In [ ]:
import glob
import matplotlib.pyplot as plt

paths = sorted(glob.glob(str(RESULTS_DIR / "*" / "spectra" / "spectra_condition_*.tsv")))
assert paths, "No condition spectra found"

path = paths[0]
spectrum = pd.read_csv(path, sep="\t")
chromosome = "17"
plot_data = spectrum[
    (spectrum["chr"].astype(str) == chromosome) &
    (spectrum["sample"] == "median_signal")
].sort_values("period")

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(plot_data["period"], plot_data["power"], lw=1)
ax.set(
    xscale="log",
    xlabel="Periodo (genes por ciclo)",
    ylabel="Potencia",
    title=f"{Path(path).stem} · chr{chromosome} · señal mediana",
)
ax.invert_xaxis()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 6 · Ventana espectral y consenso característico

In [ ]:
# Qué estructura puede inducir por sí solo el patrón de genes ausentes.
# Se mira ANTES de leer nada en un pico.
tsf("window", *DATASETS)

In [ ]:
tsf("consensus", *DATASETS, "--n-null", str(N_NULL))

In [ ]:
signature_paths = sorted(glob.glob(str(RESULTS_DIR / "**" / "signature_*.tsv"), recursive=True))
print("Signature files:", len(signature_paths))

summary_rows = []
for path in signature_paths:
    frame = pd.read_csv(path, sep="\t")
    summary_rows.append({
        "file": str(Path(path).relative_to(RESULTS_DIR)),
        "components": len(frame),
        "confirmed": int((frame.get("signature_class", "") == "confirmed").sum()),
        "exploratory": int((frame.get("signature_class", "") == "exploratory").sum()),
    })
display(pd.DataFrame(summary_rows))

## 7 · Baselines de clasificación (opcional)

Estos modelos no reemplazan el análisis estadístico del espectro característico. Sirven como referencia mínima para evaluar si una futura capa aprendida aporta algo y para mostrar qué clases pueden evaluarse realmente fuera de cohorte.

In [ ]:
tsf("ae-prepare", *DATASETS)
subprocess.run([
    sys.executable, "scripts/run_baselines.py",
    "--data", str(RESULTS_DIR / "autoencoder" / "data"),
    "--out", str(RESULTS_DIR / "autoencoder" / "baselines"),
], check=True)

class_report = pd.read_csv(
    RESULTS_DIR / "autoencoder" / "baselines" / "class_evaluation.tsv", sep="\t")
display(class_report)

## 8 · Exportar resultados

Descarga un ZIP con tablas, manifiestos y figuras. Los archivos crudos GEO no se incluyen.

In [ ]:
archive = shutil.make_archive("/content/TissueSpectF_results", "zip", RESULTS_DIR)
print("Created:", archive)

# En Google Colab, descomenta para descargarlo:
# from google.colab import files
# files.download(archive)

## Para un análisis final

Antes de interpretar biología:

- fija `REPO_REF` a un tag o commit, no a `main`;
- `FAST_MODE = False` y las permutaciones de producción (`RUNBOOK.md`, §6);
- corre `maxt`, que este notebook omite;
- conserva `manifest.tsv`, `label_audit.tsv`, `grid_coverage.tsv` y
  `count_column_map.tsv`;
- exige replicación entre cohortes y reporta heterogeneidad;
- distingue `confirmed` de `exploratory`;
- usa `peaks` para saber qué genes sostienen cada componente;
- como fondo de enriquecimiento, solo genes evaluables en la rejilla.

El modo rápido demuestra que el software corre y que las etiquetas son las que
crees. No demuestra que un pico sea biológico ni que un clasificador generalice
a una cohorte nueva.